# 10 — Network Metrics vs Senate Election Results

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("10_network_metrics_vs_election_results", total_steps=9, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook compares Twitter/X communication and network metrics with actual Senate election outcomes. It does not claim Twitter predicts votes; it tests whether online visibility and network position align with actual vote results.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
from scipy.stats import spearmanr, pearsonr, kendalltau

national = pd.read_csv(PROCESSED / "senate_national_candidate_results.csv")
mention_freq = pd.read_csv(PROCESSED / "candidate_mention_frequency.csv") if (PROCESSED / "candidate_mention_frequency.csv").exists() else pd.DataFrame(columns=["candidate", "count"])
engagement = pd.read_csv(PROCESSED / "candidate_engagement_summary.csv") if (PROCESSED / "candidate_engagement_summary.csv").exists() else pd.DataFrame(columns=["candidate"])
comention_metrics = pd.read_csv(PROCESSED / "candidate_comention_node_metrics.csv") if (PROCESSED / "candidate_comention_node_metrics.csv").exists() else pd.DataFrame(columns=["node"])
bip_metrics = pd.read_csv(PROCESSED / "candidate_hashtag_bipartite_node_metrics.csv") if (PROCESSED / "candidate_hashtag_bipartite_node_metrics.csv").exists() else pd.DataFrame(columns=["node", "node_type"])
diversity = pd.read_csv(PROCESSED / "candidate_hashtag_diversity.csv") if (PROCESSED / "candidate_hashtag_diversity.csv").exists() else pd.DataFrame(columns=["candidate"])

base = national[["candidate", "party", "total_votes", "vote_rank", "vote_share_of_valid_votes", "winner_top12"]].copy()
base = base.merge(mention_freq.rename(columns={"count": "twitter_mention_count"}), on="candidate", how="left")
if not engagement.empty:
    base = base.merge(engagement, on="candidate", how="left", suffixes=("", "_eng"))
if not comention_metrics.empty:
    cm = comention_metrics.rename(columns={"node": "candidate"}).add_prefix("comention_")
    cm = cm.rename(columns={"comention_candidate": "candidate"})
    base = base.merge(cm, on="candidate", how="left")
if not bip_metrics.empty:
    bm = bip_metrics[bip_metrics.get("node_type", "") == "candidate"].rename(columns={"node": "candidate"}).add_prefix("bipartite_")
    bm = bm.rename(columns={"bipartite_candidate": "candidate"})
    base = base.merge(bm, on="candidate", how="left")
if not diversity.empty:
    base = base.merge(diversity, on="candidate", how="left")

for c in base.columns:
    if c not in ["candidate", "party", "winner_top12"]:
        base[c] = pd.to_numeric(base[c], errors="coerce")
base = base.fillna(0)
base.to_csv(PROCESSED / "candidate_online_vs_election_metrics.csv", index=False)
base.sort_values("vote_rank").head(20)


In [ ]:
progress.step("Purpose")
# Create ranks for online metrics
online_metric_candidates = [
    "twitter_mention_count", "total_engagement", "viewCount", "retweetCount", "likeCount", "quoteCount",
    "comention_pagerank", "comention_weighted_degree", "comention_betweenness_centrality",
    "bipartite_pagerank", "bipartite_weighted_degree", "bipartite_betweenness_centrality",
    "unique_hashtags", "total_candidate_hashtag_weight"
]
online_metric_candidates = [c for c in online_metric_candidates if c in base.columns]
comparison = add_rank_columns(base, online_metric_candidates, higher_is_better=True, suffix="rank")

# Online overperformance: actual vote rank - online rank.
for metric in online_metric_candidates:
    rank_col = f"{metric}_rank"
    if rank_col in comparison.columns:
        comparison[f"{metric}_overperformance"] = comparison["vote_rank"] - comparison[rank_col]

comparison.to_csv(PROCESSED / "candidate_rank_comparison_online_vs_votes.csv", index=False)
comparison.sort_values("vote_rank").head(20)


In [ ]:
progress.step("Purpose")
# Correlation tests against vote rank and vote count
correlation_rows = []
for metric in online_metric_candidates:
    valid = comparison[[metric, "total_votes", "vote_rank"]].replace([np.inf, -np.inf], np.nan).dropna()
    valid = valid[valid[metric] > 0]
    if len(valid) >= 3:
        sp = spearmanr(valid[metric], valid["total_votes"])
        pr = pearsonr(valid[metric], valid["total_votes"])
        kt = kendalltau(valid[metric], valid["total_votes"])
        correlation_rows.append({
            "online_metric": metric,
            "n_candidates_with_metric": len(valid),
            "spearman_r_vs_votes": sp.statistic,
            "spearman_p": sp.pvalue,
            "pearson_r_vs_votes": pr.statistic,
            "pearson_p": pr.pvalue,
            "kendall_tau_vs_votes": kt.statistic,
            "kendall_p": kt.pvalue,
        })

corr = pd.DataFrame(correlation_rows).sort_values("spearman_r_vs_votes", ascending=False)
corr.to_csv(TABLES / "10_correlation_online_metrics_vs_votes.csv", index=False)
corr


In [ ]:
progress.step("Purpose")
# Top-12 overlap and precision@12 for each online metric
actual_winners = set(comparison.query("winner_top12 == True")["candidate"])
overlap_rows = []
for metric in online_metric_candidates:
    top_online = set(comparison.sort_values(metric, ascending=False).head(12)["candidate"])
    overlap = actual_winners & top_online
    overlap_rows.append({
        "online_metric": metric,
        "top12_overlap_count": len(overlap),
        "precision_at_12": len(overlap) / 12,
        "recall_at_12": len(overlap) / len(actual_winners) if actual_winners else np.nan,
        "overlapping_candidates": ", ".join(sorted(overlap)),
    })

overlap_df = pd.DataFrame(overlap_rows).sort_values("top12_overlap_count", ascending=False)
overlap_df.to_csv(TABLES / "10_top12_overlap_precision_recall.csv", index=False)
overlap_df


In [ ]:
progress.step("if not corr.empty:")
# Correlation heatmap
if not corr.empty:
    heat = corr.set_index("online_metric")[["spearman_r_vs_votes", "pearson_r_vs_votes", "kendall_tau_vs_votes"]]
    plt.figure(figsize=(10, max(6, len(heat) * .45)))
    sns.heatmap(heat, annot=True, cmap="vlag", center=0, fmt=".2f")
    plt.title("Correlation of online metrics with actual vote totals")
    plt.tight_layout()
    plt.savefig(FIGURES / "10_online_metrics_vote_correlation_heatmap.png", dpi=220, bbox_inches="tight")
    plt.show()


In [ ]:
progress.step("selected_metrics = [m for m in ['twitter_mention_count', 'comention_pagerank', 'bipartite_pagerank', 'total_engagement']")
# Vote rank vs online rank scatter using selected metrics
selected_metrics = [m for m in ["twitter_mention_count", "comention_pagerank", "bipartite_pagerank", "total_engagement"] if m in comparison.columns]
for metric in selected_metrics:
    rank_col = f"{metric}_rank"
    if rank_col in comparison.columns:
        fig = px.scatter(
            comparison,
            x="vote_rank", y=rank_col,
            color="winner_top12", hover_name="candidate", size=metric,
            title=f"Actual vote rank vs online rank: {metric}",
            labels={"vote_rank": "Actual vote rank", rank_col: f"Online rank by {metric}"}
        )
        fig.add_shape(type="line", x0=1, y0=1, x1=66, y1=66, line=dict(dash="dash"))
        fig.update_yaxes(autorange="reversed")
        fig.update_xaxes(autorange="reversed")
        fig.update_layout(height=620)
        save_plotly(fig, INTERACTIVE / f"10_vote_rank_vs_{metric}_rank.html", FIGURES / f"10_vote_rank_vs_{metric}_rank.png")
        fig.show()


In [ ]:
progress.step("main_metric = 'twitter_mention_count' if 'twitter_mention_count' in comparison.columns else (selected_metrics[0] if sele")
# Online overperformance chart for the main online metric available
main_metric = "twitter_mention_count" if "twitter_mention_count" in comparison.columns else (selected_metrics[0] if selected_metrics else None)
if main_metric:
    over_col = f"{main_metric}_overperformance"
    if over_col in comparison.columns:
        plot_df = comparison.sort_values(over_col)
        fig = px.bar(plot_df, x=over_col, y="candidate", orientation="h", color="winner_top12", title=f"Online overperformance index based on {main_metric}")
        fig.update_layout(height=1100, yaxis_title="Candidate", xaxis_title="Vote rank - online rank")
        save_plotly(fig, INTERACTIVE / "10_online_overperformance_index.html", FIGURES / "10_online_overperformance_index.png")
        fig.show()


In [ ]:
progress.step("melt_cols = [c for c in selected_metrics if c in comparison.columns]")
# Winner vs non-winner comparison for available metrics
melt_cols = [c for c in selected_metrics if c in comparison.columns]
if melt_cols:
    melted = comparison.melt(id_vars=["candidate", "winner_top12"], value_vars=melt_cols, var_name="metric", value_name="value")
    melted["log1p_value"] = np.log1p(melted["value"])
    plt.figure(figsize=(14, 7))
    sns.boxplot(data=melted, x="metric", y="log1p_value", hue="winner_top12")
    plt.title("Winner vs non-winner comparison across online metrics")
    plt.xlabel("Online metric")
    plt.ylabel("log(1 + metric value)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURES / "10_winner_nonwinner_online_metric_boxplots.png", dpi=220, bbox_inches="tight")
    plt.show()


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
